# Yellow-trip hourly TPP split artifacts

이 노트북은 `sample_data/new_york_taxi/yellow_trip_hourly.parquet`를 quantity reconstruction 실험용 `train / validation / test` parquet로 분리합니다.

`yellow_trip_hourly.parquet`은 raw taxi trip log가 아니라, `yellow_trip.ipynb`에서 미리 생성한 hourly grid-cell event table입니다. 여기서는 해당 event table에 chronological split과 magnitude-factorized quantity labels를 추가합니다.

## 1. Setup

노트북 실행 위치가 project root 또는 notebook 폴더 어디든 동일하게 동작하도록 import path를 정리합니다.

In [2]:
from pathlib import Path
import sys

def resolve_project_root(start: Path | None = None) -> Path:
    """Find the project root even when Jupyter starts from home or /tmp."""
    anchor = (start or Path.cwd()).expanduser().resolve()
    base = anchor if anchor.is_dir() else anchor.parent
    candidates = [base, *base.parents]
    candidates.extend([
        Path('~/workspace/paper_research').expanduser(),
        Path('/home/workspace/paper_research'),
        Path('/Users/igwanhyeong/PycharmProjects/paper_research'),
    ])
    for candidate in candidates:
        if all((candidate / name).exists() for name in ('sample_data', 'models', 'utils')):
            return candidate
    raise RuntimeError('Could not locate the paper_research project root.')

PROJECT_ROOT = resolve_project_root()
PREPROCESSING_DIR = PROJECT_ROOT / 'simple_lab_test' / 'notebooks' / 'preprocessing'
for path in (PROJECT_ROOT, PREPROCESSING_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from tpp_split_utils import SplitConfig, build_and_save_quantity_splits, ensure_project_root_on_path

PROJECT_ROOT = ensure_project_root_on_path(PROJECT_ROOT)
print('PROJECT_ROOT =', PROJECT_ROOT)

PROJECT_ROOT = /home/leekwanhyeong/workspace/paper_research


## 2. Split configuration

Yellow-trip hourly는 quantity tail이 크고 hourly grid-cell별 sequence가 길기 때문에 기존 실험 정책과 동일하게 `scale_base=10.0`을 사용합니다.

In [3]:
cfg = SplitConfig(
    dataset_name='yellow_trip_hourly',
    input_path=PROJECT_ROOT / 'sample_data' / 'new_york_taxi' / 'yellow_trip_hourly.parquet',
    output_dir=PROJECT_ROOT / 'sample_data' / 'new_york_taxi',
    output_prefix='yellow_trip_hourly',
    scale_base=10.0,
    train_ratio=0.70,
    validation_ratio=0.15,
    test_ratio=0.15,
)
cfg

SplitConfig(dataset_name='yellow_trip_hourly', input_path=PosixPath('/home/leekwanhyeong/workspace/paper_research/sample_data/new_york_taxi/yellow_trip_hourly.parquet'), output_dir=PosixPath('/home/leekwanhyeong/workspace/paper_research/sample_data/new_york_taxi'), output_prefix='yellow_trip_hourly', scale_base=10.0, train_ratio=0.7, validation_ratio=0.15, test_ratio=0.15, entity_col='oper_part_no', order_col='seq', clip_min_qty=1.0, min_count=100, min_coverage=0.999)

## 3. Build and save artifacts

이 셀은 train 기준으로 magnitude upper-order cap을 정한 뒤 전체 event table에 동일한 mark/residual rule을 적용합니다.

In [4]:
result = build_and_save_quantity_splits(cfg, overwrite=True)
print('max_order fitted on train =', result['max_order'])
for name, path in result['paths'].items():
    print(f'{name}: {path}')

max_order fitted on train = 3
with_split: /home/leekwanhyeong/workspace/paper_research/sample_data/new_york_taxi/yellow_trip_hourly_with_split.parquet
train: /home/leekwanhyeong/workspace/paper_research/sample_data/new_york_taxi/yellow_trip_hourly_train.parquet
validation: /home/leekwanhyeong/workspace/paper_research/sample_data/new_york_taxi/yellow_trip_hourly_validation.parquet
test: /home/leekwanhyeong/workspace/paper_research/sample_data/new_york_taxi/yellow_trip_hourly_test.parquet
manifest: /home/leekwanhyeong/workspace/paper_research/sample_data/new_york_taxi/yellow_trip_hourly_split_manifest.json


## 4. Quality checks

split별 row 수, series 수, mark 분포를 확인합니다. Yellow-trip은 series 수가 적고 sequence가 길기 때문에 split 비율이 비교적 안정적으로 나뉘어야 합니다.

In [5]:
from IPython.display import display
import polars as pl

labeled_df = result['labeled_df']
display(labeled_df.head())
display(pl.DataFrame(result['summary']['split_counts']))
display(pl.DataFrame(result['summary']['mark_counts']).pivot(values='len', index='mark', columns='chronological_split').fill_null(0).sort('mark'))
display(labeled_df.select([
    pl.len().alias('rows'),
    pl.col('oper_part_no').n_unique().alias('series'),
    pl.col('demand_qty').median().alias('qty_median'),
    pl.col('demand_qty').quantile(0.95).alias('qty_p95'),
    pl.col('demand_qty').max().alias('qty_max'),
    pl.col('scale_residual').min().alias('residual_min'),
    pl.col('scale_residual').max().alias('residual_max'),
]))

oper_part_no,demand_dt,seq,delta_t,demand_qty,time_bucket,source_resolution,grid_size_deg,min_active_buckets,chronological_split,log_qty,log10_qty,raw_order,mark,scale_residual,z
str,i64,i64,i32,f64,datetime[μs],str,f64,i32,str,f64,f64,i32,i32,f64,f64
"""-3677_2038""",20150101120000,13,0,1.0,2015-01-01 12:00:00,"""hourly""",0.02,72,"""train""",0.0,0.0,0,0,0.0,0.0
"""-3677_2038""",20150101130000,14,1,1.0,2015-01-01 13:00:00,"""hourly""",0.02,72,"""train""",0.0,0.0,0,0,0.0,0.0
"""-3677_2038""",20150101140000,15,1,2.0,2015-01-01 14:00:00,"""hourly""",0.02,72,"""train""",0.30103,0.30103,0,0,0.30103,0.30103
"""-3677_2038""",20150101150000,16,1,1.0,2015-01-01 15:00:00,"""hourly""",0.02,72,"""train""",0.0,0.0,0,0,0.0,0.0
"""-3677_2038""",20150101160000,17,1,2.0,2015-01-01 16:00:00,"""hourly""",0.02,72,"""train""",0.30103,0.30103,0,0,0.30103,0.30103


chronological_split,rows,series,qty_median,qty_p95,qty_max
str,i64,i64,f64,f64,f64
"""test""",8327,131,7.0,1577.0,5952.0
"""train""",38524,131,7.0,1562.0,6322.0
"""validation""",8268,131,6.0,1422.0,6489.0


/tmp/ipykernel_1260720/3830830770.py:7: DeprecationWarning: the argument `columns` for `DataFrame.pivot` is deprecated. It was renamed to `on` in version 1.0.0.
  display(pl.DataFrame(result['summary']['mark_counts']).pivot(values='len', index='mark', columns='chronological_split').fill_null(0).sort('mark'))


mark,test,train,validation
i64,i64,i64,i64
0,4516,20591,4577
1,2026,9583,2008
2,1158,5489,1122
3,627,2861,561


rows,series,qty_median,qty_p95,qty_max,residual_min,residual_max
u32,u32,f64,f64,f64,f64,f64
55119,131,7.0,1547.0,6489.0,0.0,1.0
